# Taller 1 — Procesamiento de datos olímpicos con Apache Spark

**Curso:** Gestión de datos y tecnologías Big Data
**Grupo:** 1
**Integrantes:** Yerko Gallardo, Sebastián, José Ignacio Rodríguez

**Objetivo:** procesar el registro histórico de deportistas y resultados olímpicos con PySpark, utilizando RDDs, DataFrames y Spark SQL.

**Fuente de datos:** archivos entregados en el curso, ubicados en la carpeta `data/` del proyecto (`deportista.csv`, `deportista2.csv`, `resultados.csv`, `evento.csv`, `equipo.csv`, `juegos.json`).

## 1. Configuración del entorno

El proyecto se gestiona con **uv**: la versión de Python y los paquetes quedan fijados en `pyproject.toml`, `uv.lock` y `.python-version`, lo que permite reproducir el mismo entorno en cualquier equipo.
**Java** es una dependencia de ejecución de Spark (el motor corre en la JVM) y se instala fuera de uv.

| Componente | Versión |
|---|---|
| Python | 3.14 |
| PySpark | 4.3.0 |
| Java (JDK) | 21 (Eclipse Temurin) |

**Preparación (macOS):**

```bash
# Java
brew install --cask temurin@21
echo 'export JAVA_HOME=$(/usr/libexec/java_home -v 21)' >> ~/.zshrc
source ~/.zshrc
java -version

# Proyecto
git clone https://github.com/joserodriguezc/gestiondatosytecnologiasbigdata.git
cd gestiondatosytecnologiasbigdata
uv sync --locked --no-install-project

# Editor
code .        # VS Code: seleccionar el kernel .venv (Python 3.14)
# alternativa: uv run --no-sync jupyter lab
```

La sesión se crea en modo clásico (no Spark Connect), ya que el taller requiere RDDs y `SparkContext`.

En Linux, consultar `README.md`: esta ejecución selecciona el JDK 17 instalado mediante `JAVA_HOME`. La tabla anterior describe la preparación original de Yerko; las versiones efectivas se imprimen abajo. `--no-install-project` evita intentar construir el módulo ausente de este repositorio de notebooks.


### 1.1 Verificación del intérprete, versiones y Java

In [1]:
import os, sys, platform, subprocess
import pyspark

# Si el editor no heredó JAVA_HOME, se obtiene del sistema (macOS) sin fijar rutas personales
if not os.environ.get("JAVA_HOME") and platform.system() == "Darwin":
    os.environ["JAVA_HOME"] = subprocess.run(
        ["/usr/libexec/java_home", "-v", "21"], capture_output=True, text=True
    ).stdout.strip()

print("Sistema    :", platform.system(), platform.machine())
print("Intérprete :", sys.executable)
print("Python     :", platform.python_version())
print("PySpark    :", pyspark.__version__)
print("JAVA_HOME  :", os.environ.get("JAVA_HOME", "NO DEFINIDO"))

java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java       :")
print(java.stderr.strip())   # java -version escribe en stderr

Sistema    : Linux x86_64
Intérprete : /home/sebastian/Escritorio/magister/proyectos/gestiondatosytecnologiasbigdata/.venv/bin/python
Python     : 3.14.6
PySpark    : 4.3.0
JAVA_HOME  : /usr/lib/jvm/java-17-openjdk
Java       :
openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1, mixed mode, sharing)


### 1.2 SparkSession y SparkContext

`PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` apuntan al intérprete del proyecto para que los procesos Python de Spark usen el mismo entorno.

In [2]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Taller1_Spark")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

print("Tipo de sesión     :", type(spark).__module__)   # pyspark.sql.session = modo clásico
print("Versión Spark      :", spark.version)
print("Master             :", sc.master)
print("Nombre aplicación  :", sc.appName)
print("Paralelismo default:", sc.defaultParallelism)
print("Spark UI           :", sc.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/03 22:17:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Tipo de sesión     : pyspark.sql.session
Versión Spark      : 4.3.0
Master             : local[*]
Nombre aplicación  : Taller1_Spark
Paralelismo default: 12
Spark UI           : http://localhost:4040


### 1.3 Prueba mínima de RDD y DataFrame

In [3]:
rdd_prueba = sc.parallelize(range(1, 11), 2)
print("RDD -> suma:", rdd_prueba.sum(), "| particiones:", rdd_prueba.getNumPartitions())

df_prueba = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "letra"])
df_prueba.show()
df_prueba.printSchema()

RDD -> suma: 55 | particiones: 2


+---+-----+
| id|letra|
+---+-----+
|  1|    a|
|  2|    b|
+---+-----+

root
 |-- id: long (nullable = true)
 |-- letra: string (nullable = true)



## 2. Lectura e inspección de los archivos de deportistas

Los datos de deportistas vienen repartidos en dos archivos CSV con la misma estructura de siete columnas:

| Columna | Contenido | Tipo |
|---|---|---|
| `deportista_id` | identificador del deportista (clave usada en `resultados.csv`) | entero |
| `nombre` | nombre del deportista | texto |
| `genero` | 1 = hombre, 2 = mujer | entero |
| `edad` | edad en años | entero |
| `altura` | altura en centímetros | decimal |
| `peso` | peso en kilogramos | decimal |
| `equipo_id` | identificador del equipo (clave de `equipo.csv`) | entero |

Antes de crear los RDDs se revisa el contenido crudo de cada archivo: separador, presencia de encabezado, uso de comillas y valores ausentes.

### 2.1 Ubicación de los archivos

In [4]:
from pathlib import Path

DIR_DATOS = Path("data")
RUTA_DEPORTISTA = DIR_DATOS / "deportista.csv"
RUTA_DEPORTISTA2 = DIR_DATOS / "deportista2.csv"

for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    estado = f"{ruta.stat().st_size:,} bytes" if ruta.exists() else "NO ENCONTRADO"
    print(f"{str(ruta):22s} {estado}")

faltantes = [str(r) for r in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2) if not r.exists()]
if faltantes:
    raise FileNotFoundError(f"Faltan archivos de entrada en la carpeta data/: {faltantes}")

data/deportista.csv    2,832,265 bytes
data/deportista2.csv   2,874,875 bytes


### 2.2 Contenido crudo

Se muestran las primeras líneas tal como las lee Spark con `textFile`. Se usa `repr` para dejar visibles las comillas y los caracteres de fin de línea.

In [5]:
for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    lineas = sc.textFile(str(ruta))
    print(f"--- {ruta.name}: {lineas.count():,} líneas")
    for linea in lineas.take(5):
        print("   ", repr(linea))
    print()

--- deportista.csv: 67,786 líneas
    '1,A Dijiang,1,24,180,80,199'
    '2,A Lamusi,1,23,170,60,199'
    '3,Gunnar Nielsen Aaby,1,24,0,0,273'
    '4,Edgar Lindenau Aabye,1,34,0,0,278'
    '5,Christine Jacoba Aaftink,2,21,185,82,705'

--- deportista2.csv: 67,785 líneas
    '67787,Lee BongJu,1,27,167,56,970'
    '67788,Lee BuTi,1,23,164,54,203'
    '67789,Anthony N. Buddy Lee,1,34,172,62,1096'
    '67790,Alfred A. Butch Lee Porter,1,19,186,80,825'
    '67791,Lee ByeongGu,1,22,175,68,970'



### 2.3 Reglas de lectura

- **Separación de campos:** cada línea se divide con el lector CSV de Python, que respeta los nombres entre comillas aunque contengan comas.
- **Encabezado:** solo la primera línea de un archivo puede ser encabezado; se reconoce porque su primer campo no es numérico. Se retira y se informa.
- **Valores ausentes:** los campos vacíos o con `NA` se interpretan como desconocidos (`None`).
- **Campos vacíos sobrantes:** si una línea trae más de siete campos y los sobrantes están vacíos (coma final de más), se conservan los siete primeros. Estos casos se cuentan y se muestran.
- **Registros inválidos:** una línea con un número de campos distinto de siete o con valores numéricos imposibles de convertir no se descarta en silencio: se separa en un RDD de inválidos y se informa con su motivo.
- **Ceros:** se conservan tal como vienen en los RDDs para mantener la trazabilidad con la fuente; su tratamiento se define al construir el DataFrame.

In [6]:
import csv

COLUMNAS = ["deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"]
VALORES_AUSENTES = {"", "NA", "NaN", "null", "None"}


def separar_campos(linea):
    """Divide una línea CSV respetando comillas (nombres que contienen comas)."""
    return next(csv.reader([linea.rstrip("\r")]))


def sobran_campos_vacios(campos):
    """True si la línea trae más de siete campos y los sobrantes están vacíos (coma final extra)."""
    return len(campos) > len(COLUMNAS) and all(c.strip() == "" for c in campos[len(COLUMNAS):])


def normalizar_campos(campos):
    """Elimina los campos vacíos sobrantes al final de la línea; el resto queda igual."""
    return campos[:len(COLUMNAS)] if sobran_campos_vacios(campos) else campos


def es_encabezado(campos):
    return len(campos) > 0 and not campos[0].strip().isdigit()


def a_entero(valor):
    valor = valor.strip()
    if valor in VALORES_AUSENTES:
        return None
    numero = float(valor)
    if not numero.is_integer():
        raise ValueError(f"'{valor}' no es entero")
    return int(numero)


def a_decimal(valor):
    valor = valor.strip()
    return None if valor in VALORES_AUSENTES else float(valor)


def convertir_registro(campos):
    """Devuelve (registro_tipado, None) si la línea es válida, o (campos, motivo) si no lo es."""
    if len(campos) != len(COLUMNAS):
        return campos, f"tiene {len(campos)} campos"
    try:
        d_id, nombre, genero, edad, altura, peso, equipo_id = campos
        registro = (a_entero(d_id), nombre.strip(), a_entero(genero), a_entero(edad),
                    a_decimal(altura), a_decimal(peso), a_entero(equipo_id))
    except ValueError as error:
        return campos, f"valor no numérico ({error})"
    if registro[0] is None:
        return campos, "deportista_id vacío"
    return registro, None


def leer_deportistas(lineas):
    """A partir de un RDD de líneas devuelve (validos, invalidos, encabezado)."""
    indexadas = lineas.zipWithIndex()
    encabezado = (indexadas
                  .filter(lambda par: par[1] == 0 and es_encabezado(separar_campos(par[0])))
                  .map(lambda par: par[0]))
    convertidos = (indexadas
                   .filter(lambda par: not (par[1] == 0 and es_encabezado(separar_campos(par[0]))))
                   .map(lambda par: convertir_registro(normalizar_campos(separar_campos(par[0])))))
    validos = convertidos.filter(lambda r: r[1] is None).map(lambda r: r[0])
    invalidos = convertidos.filter(lambda r: r[1] is not None)
    return validos, invalidos, encabezado

### 2.4 Diagnóstico de calidad de cada archivo

Para cada archivo se cuenta el encabezado, los registros válidos e inválidos, los valores ausentes y los ceros de las columnas numéricas, y la codificación de `genero`.

In [7]:
def diagnosticar(ruta):
    lineas = sc.textFile(str(ruta))
    validos, invalidos, encabezado = leer_deportistas(lineas)
    validos.cache()

    total_lineas = lineas.count()
    n_encabezado = encabezado.count()
    n_validos = validos.count()
    n_invalidos = invalidos.count()

    print(f"=== {ruta.name}")
    print(f"Líneas leídas       : {total_lineas:,}")
    print(f"Encabezado          : {encabezado.first() if n_encabezado else 'no tiene'}")
    reparadas = lineas.filter(lambda l: sobran_campos_vacios(separar_campos(l)))
    print(f"Registros válidos   : {n_validos:,}")
    print(f"  con campos vacíos sobrantes reparados: {reparadas.count():,}")
    for linea in reparadas.take(5):
        print("    ", repr(linea))
    print(f"Registros inválidos : {n_invalidos:,}")
    for campos, motivo in invalidos.take(5):
        print(f"    {motivo}: {campos}")
    print(f"Conciliación        : {n_encabezado} + {n_validos:,} + {n_invalidos:,} = "
          f"{n_encabezado + n_validos + n_invalidos:,} ({'OK' if n_encabezado + n_validos + n_invalidos == total_lineas else 'NO CUADRA'})")

    ids = validos.map(lambda r: r[0])
    print(f"Rango deportista_id : {ids.min():,} a {ids.max():,} | ids distintos: {ids.distinct().count():,}")

    print(f"{'columna':10s} {'ausentes':>9s} {'ceros':>9s} {'mínimo':>8s} {'máximo':>8s}")
    for i, columna in enumerate(COLUMNAS):
        if columna in ("deportista_id", "nombre"):
            continue
        valores = validos.map(lambda r, i=i: r[i])
        presentes = valores.filter(lambda v: v is not None)
        print(f"{columna:10s} {valores.filter(lambda v: v is None).count():9,} "
              f"{presentes.filter(lambda v: v == 0).count():9,} {presentes.min():8} {presentes.max():8}")

    print("Codificación genero :", dict(sorted(validos.map(lambda r: r[2]).countByValue().items(),
                                              key=lambda kv: (kv[0] is None, kv[0]))))
    con_comillas = lineas.filter(lambda l: '"' in l).count()
    print(f"Líneas con comillas : {con_comillas:,}")
    validos.unpersist()
    print()


for ruta in (RUTA_DEPORTISTA, RUTA_DEPORTISTA2):
    diagnosticar(ruta)

=== deportista.csv
Líneas leídas       : 67,786
Encabezado          : no tiene
Registros válidos   : 67,786
  con campos vacíos sobrantes reparados: 1


     '16587,Vladimr Bil,1,24,191,92,67,'
Registros inválidos : 0


Conciliación        : 0 + 67,786 + 0 = 67,786 (OK)


Rango deportista_id : 1 a 67,786 | ids distintos: 67,786
columna     ausentes     ceros   mínimo   máximo
genero             0         0        1        2


edad               0     3,235        0       96
altura             0    17,435      0.0    223.0


peso               0    17,949      0.0    214.0
equipo_id          0        64        0     1184
Codificación genero : {1: 51380, 2: 16406}


... (3 outputs left out)

Conciliación        : 0 + 67,785 + 0 = 67,785 (OK)
Rango deportista_id : 67,787 a 135,571 | ids distintos: 67,785
columna     ausentes     ceros   mínimo   máximo


genero             0         0        1        2
edad               0     3,133        0       97


altura             0    16,481      0.0    226.0
peso               0    16,936      0.0    198.0


equipo_id          0        79        0     1184
Codificación genero : {1: 50210, 2: 17575}
Líneas con comillas : 0



**Hallazgos de la inspección**

| | `deportista.csv` | `deportista2.csv` |
|---|---|---|
| Líneas | 67.786 | 67.785 |
| Encabezado | no tiene | no tiene |
| Fin de línea | CRLF (Windows) | LF |
| Rango de `deportista_id` | 1 a 67.786 | 67.787 a 135.571 |
| Registros inválidos | 0 | 0 |
| Campos vacíos sobrantes reparados | 1 | 0 |
| Edad = 0 | 3.235 | 3.133 |
| Altura = 0 | 17.435 | 16.481 |
| Peso = 0 | 17.949 | 16.936 |
| `equipo_id` = 0 | 64 | 79 |
| Género 1 / 2 | 51.380 / 16.406 | 50.210 / 17.575 |

- Ninguno de los dos archivos trae encabezado y ninguno usa comillas; aun así, el lector respeta comillas por si un nombre contuviera comas. El fin de línea CRLF de `deportista.csv` se elimina al leer.
- Los dos archivos son complementarios: los identificadores son continuos del 1 al 135.571, sin repetidos ni vacíos entre ellos.
- La línea del deportista **16587** (`16587,Vladimr Bil,1,24,191,92,67,`) termina con una coma de más, que genera un octavo campo vacío. El registro está completo, por lo que se conservan sus siete campos en lugar de descartarlo.
- `genero` usa exclusivamente los códigos 1 (hombre) y 2 (mujer).
- Hay valores **0** en `edad`, `altura` y `peso`. Una edad, altura o peso igual a cero no es posible en un deportista: se trata de datos no registrados en la fuente. En los RDDs se conservan sin cambios y en el DataFrame se convierten a nulo.
- 143 deportistas tienen `equipo_id = 0`, código que no existe en `equipo.csv`; se mantienen y se informan en la integración con las demás tablas.
- Algunos nombres muestran caracteres perdidos en la fuente (por ejemplo, `Vladimr Bil` o `Piotr ya`, donde faltan letras con diacríticos); se conservan tal como vienen.

## 3. Creación de RDDs y unión

Se crea el RDD `deportista` a partir de `deportista.csv` con **seis particiones** y el RDD `deportista2` a partir de `deportista2.csv`. Luego se unen en `deportistaTotal`.

`textFile` recibe un número **mínimo** de particiones; si Spark genera más de seis, se reducen con `coalesce(6)`, que no redistribuye los datos entre nodos.

In [8]:
lineas_deportista = sc.textFile(str(RUTA_DEPORTISTA), minPartitions=6)
deportista, invalidos_deportista, encabezado_deportista = leer_deportistas(lineas_deportista)
if deportista.getNumPartitions() > 6:
    deportista = deportista.coalesce(6)
deportista = deportista.cache()

print("RDD deportista")
print("  Particiones          :", deportista.getNumPartitions())
print("  Registros por partición:", deportista.glom().map(len).collect())
print(f"  Registros válidos    : {deportista.count():,}")
print(f"  Registros inválidos  : {invalidos_deportista.count():,}")
print("  Muestra:")
for registro in deportista.take(5):
    print("   ", registro)

RDD deportista
  Particiones          : 6
  Registros por partición: [11396, 11236, 11095, 11243, 11424, 11392]
  Registros válidos    : 67,786


  Registros inválidos  : 0
  Muestra:
    (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
    (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
    (3, 'Gunnar Nielsen Aaby', 1, 24, 0.0, 0.0, 273)
    (4, 'Edgar Lindenau Aabye', 1, 34, 0.0, 0.0, 278)
    (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)


In [9]:
lineas_deportista2 = sc.textFile(str(RUTA_DEPORTISTA2))
deportista2, invalidos_deportista2, encabezado_deportista2 = leer_deportistas(lineas_deportista2)
deportista2 = deportista2.cache()

print("RDD deportista2")
print("  Particiones          :", deportista2.getNumPartitions())
print(f"  Registros válidos    : {deportista2.count():,}")
print(f"  Registros inválidos  : {invalidos_deportista2.count():,}")
print("  Muestra:")
for registro in deportista2.take(5):
    print("   ", registro)

RDD deportista2
  Particiones          : 2


  Registros válidos    : 67,785


  Registros inválidos  : 0
  Muestra:
    (67787, 'Lee BongJu', 1, 27, 167.0, 56.0, 970)
    (67788, 'Lee BuTi', 1, 23, 164.0, 54.0, 203)
    (67789, 'Anthony N. Buddy Lee', 1, 34, 172.0, 62.0, 1096)
    (67790, 'Alfred A. Butch Lee Porter', 1, 19, 186.0, 80.0, 825)
    (67791, 'Lee ByeongGu', 1, 22, 175.0, 68.0, 970)


### 3.1 Unión en `deportistaTotal`

`union` concatena ambos RDDs sin eliminar duplicados, por lo que el número de particiones del resultado es la suma de las particiones de ambos. Se verifica que el total cuadre con la suma de los registros válidos de cada archivo y que ningún `deportista_id` se repita entre los dos archivos.

In [10]:
deportistaTotal = deportista.union(deportista2).cache()

n_deportista = deportista.count()
n_deportista2 = deportista2.count()
n_total = deportistaTotal.count()
n_ids_distintos = deportistaTotal.map(lambda r: r[0]).distinct().count()

print("RDD deportistaTotal")
print("  Particiones              :", deportistaTotal.getNumPartitions(),
      f"({deportista.getNumPartitions()} + {deportista2.getNumPartitions()})")
print(f"  deportista + deportista2 : {n_deportista:,} + {n_deportista2:,} = {n_deportista + n_deportista2:,}")
print(f"  deportistaTotal          : {n_total:,}")
print(f"  deportista_id distintos  : {n_ids_distintos:,}")

assert n_total == n_deportista + n_deportista2, "La unión no conserva todos los registros"
print("  Conciliación             : OK" if n_ids_distintos == n_total
      else f"  Atención: {n_total - n_ids_distintos:,} identificadores repetidos entre archivos")

RDD deportistaTotal
  Particiones              : 8 (6 + 2)
  deportista + deportista2 : 67,786 + 67,785 = 135,571
  deportistaTotal          : 135,571
  deportista_id distintos  : 135,571
  Conciliación             : OK


## 4. Transformaciones sobre RDDs

Las transformaciones se aplican sobre `deportistaTotal` con operaciones propias de RDD (`filter`, `map`), sin pasar por DataFrames. Cada registro es una tupla con el orden de `COLUMNAS`: la edad está en la posición 3 y el género en la posición 2.

### 4.1 `MayorEdad`: deportistas de 18 años o más

Una edad igual a 0 o ausente corresponde a un dato no registrado; esos deportistas no se cuentan como mayores ni como menores de edad, y se informan por separado.

In [11]:
MayorEdad = deportistaTotal.filter(lambda r: r[3] is not None and r[3] >= 18)

edad_desconocida = deportistaTotal.filter(lambda r: r[3] is None or r[3] == 0)
menores = deportistaTotal.filter(lambda r: r[3] is not None and 0 < r[3] < 18)

n_mayores, n_menores, n_desconocida = MayorEdad.count(), menores.count(), edad_desconocida.count()

print(f"Mayores de edad (>= 18) : {n_mayores:,}")
print(f"Menores de edad (1-17)  : {n_menores:,}")
print(f"Edad no registrada      : {n_desconocida:,}")
print(f"Suma                    : {n_mayores + n_menores + n_desconocida:,} de {n_total:,}")
print("Edad mínima en MayorEdad:", MayorEdad.map(lambda r: r[3]).min())
print("Muestra:")
for registro in MayorEdad.take(5):
    print("   ", registro)

Mayores de edad (>= 18) : 123,244
Menores de edad (1-17)  : 5,959
Edad no registrada      : 6,368
Suma                    : 135,571 de 135,571


Edad mínima en MayorEdad: 18
Muestra:
    (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
    (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
    (3, 'Gunnar Nielsen Aaby', 1, 24, 0.0, 0.0, 273)
    (4, 'Edgar Lindenau Aabye', 1, 34, 0.0, 0.0, 278)
    (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)


### 4.2 `Deportistas_mujer`: deportistas con género 2

Según la codificación de la fuente, el valor 2 de `genero` corresponde a mujer.

In [12]:
Deportistas_mujer = deportistaTotal.filter(lambda r: r[2] == 2)

n_mujeres = Deportistas_mujer.count()
n_hombres = deportistaTotal.filter(lambda r: r[2] == 1).count()

print(f"Mujeres (genero = 2)        : {n_mujeres:,}")
print(f"Hombres (genero = 1)        : {n_hombres:,}")
print(f"Otro código o ausente       : {n_total - n_mujeres - n_hombres:,}")
print("Valores de genero en el RDD :", sorted(Deportistas_mujer.map(lambda r: r[2]).distinct().collect()))
print("Muestra:")
for registro in Deportistas_mujer.take(5):
    print("   ", registro)

Mujeres (genero = 2)        : 33,981
Hombres (genero = 1)        : 101,590
Otro código o ausente       : 0
Valores de genero en el RDD : [2]
Muestra:
    (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)
    (8, 'Cornelia Cor Aalten Strannood', 2, 18, 168.0, 0.0, 705)
    (13, 'Minna Maarit Aalto', 2, 30, 159.0, 55.5, 350)
    (14, 'Pirjo Hannele Aalto Mattila', 2, 32, 171.0, 65.0, 350)
    (21, 'Ragnhild Margrethe Aamodt', 2, 27, 163.0, 0.0, 742)


### 4.3 Conversión a mayúsculas

Se convierten a mayúsculas los campos de texto de `deportistaTotal`; en este conjunto el único campo textual es `nombre`. Los valores numéricos no se modifican.

Antes de reasignar el nombre se guarda una referencia al RDD original (`deportistaTotal_original`) para comparar ambos y mantener la trazabilidad. Como `map` conserva el orden y las particiones, los registros de ambos RDDs pueden emparejarse con `zip`.

In [13]:
deportistaTotal_original = deportistaTotal

deportistaTotal = deportistaTotal_original.map(
    lambda r: tuple(v.upper() if isinstance(v, str) else v for v in r)
).cache()

print(f"{'deportista_id':>13s}  {'nombre original':35s} nombre en mayúsculas")
for original, mayus in deportistaTotal_original.zip(deportistaTotal).take(10):
    print(f"{original[0]:>13}  {original[1]:35s} {mayus[1]}")

pares = deportistaTotal_original.zip(deportistaTotal)
numericos_alterados = pares.filter(lambda p: p[0][0] != p[1][0] or p[0][2:] != p[1][2:]).count()
sin_minusculas = deportistaTotal.filter(lambda r: r[1] != r[1].upper()).count()

print()
print(f"Registros en deportistaTotal       : {deportistaTotal.count():,}")
print(f"Nombres que aún tienen minúsculas  : {sin_minusculas:,}")
print(f"Registros con valores numéricos alterados: {numericos_alterados:,}")

deportista_id  nombre original                     nombre en mayúsculas


            1  A Dijiang                           A DIJIANG
            2  A Lamusi                            A LAMUSI
            3  Gunnar Nielsen Aaby                 GUNNAR NIELSEN AABY
            4  Edgar Lindenau Aabye                EDGAR LINDENAU AABYE
            5  Christine Jacoba Aaftink            CHRISTINE JACOBA AAFTINK
            6  Per Knut Aaland                     PER KNUT AALAND
            7  John Aalberg                        JOHN AALBERG
            8  Cornelia Cor Aalten Strannood       CORNELIA COR AALTEN STRANNOOD
            9  Antti Sami Aalto                    ANTTI SAMI AALTO
           10  Einar Ferdinand Einari Aalto        EINAR FERDINAND EINARI AALTO



Registros en deportistaTotal       : 135,571
Nombres que aún tienen minúsculas  : 0
Registros con valores numéricos alterados: 0


## 5. Conversión a DataFrame `deportista`

`deportistaTotal` (ya en mayúsculas) se convierte en el DataFrame `deportista` con las siete columnas y un esquema explícito, de modo que cada columna tenga un tipo numérico o de texto utilizable en las etapas siguientes.

- `edad`, `altura` y `peso` iguales a 0 se convierten a **nulo**, porque representan datos no registrados. Así no distorsionan promedios, mínimos ni el cálculo de IMC.
- `genero` se mantiene con su codificación original (1 = hombre, 2 = mujer).
- `equipo_id` se mantiene sin cambios, incluido el valor 0, para revisarlo en la integración con `equipo.csv`.

El nombre `deportista` pasa a designar el DataFrame; el RDD original se conserva en `rdd_deportista`.

In [14]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

rdd_deportista = deportista   # referencia al RDD original antes de reutilizar el nombre

esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("genero", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("altura", DoubleType(), True),
    StructField("peso", DoubleType(), True),
    StructField("equipo_id", IntegerType(), True),
])


def cero_a_nulo(valor):
    return None if valor == 0 else valor


filas = deportistaTotal.map(
    lambda r: (r[0], r[1], r[2], cero_a_nulo(r[3]), cero_a_nulo(r[4]), cero_a_nulo(r[5]), r[6])
)
deportista = spark.createDataFrame(filas, schema=esquema_deportista)

print(f"Filas del DataFrame deportista: {deportista.count():,} (deportistaTotal: {deportistaTotal.count():,})")
print("Columnas:", deportista.columns)
deportista.printSchema()
deportista.show(10, truncate=False)

Filas del DataFrame deportista: 135,571 (deportistaTotal: 135,571)
Columnas: ['deportista_id', 'nombre', 'genero', 'edad', 'altura', 'peso', 'equipo_id']
root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

+-------------+-----------------------------+------+----+------+----+---------+
|deportista_id|nombre                       |genero|edad|altura|peso|equipo_id|
+-------------+-----------------------------+------+----+------+----+---------+
|1            |A DIJIANG                    |1     |24  |180.0 |80.0|199      |
|2            |A LAMUSI                     |1     |23  |170.0 |60.0|199      |
|3            |GUNNAR NIELSEN AABY          |1     |24  |NULL  |NULL|273      |
|4            |EDGAR LINDENAU AABYE         |1     |34  |NULL  |NULL|278      |
|5 

Traceback (most recent call last):
  File "/home/sebastian/Escritorio/magister/proyectos/gestiondatosytecnologiasbigdata/.venv/lib/python3.14/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 264, in manager
    code = worker(sock, authenticated)
  File "/home/sebastian/Escritorio/magister/proyectos/gestiondatosytecnologiasbigdata/.venv/lib/python3.14/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


### 5.1 Valores nulos por columna

In [15]:
deportista.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in deportista.columns]
).show()

deportista.describe("edad", "altura", "peso").show()

+-------------+------+------+----+------+-----+---------+
|deportista_id|nombre|genero|edad|altura| peso|equipo_id|
+-------------+------+------+----+------+-----+---------+
|            0|     0|     0|6368| 33916|34885|        0|
+-------------+------+------+----+------+-----+---------+



+-------+------------------+------------------+-----------------+
|summary|              edad|            altura|             peso|
+-------+------------------+------------------+-----------------+
|  count|            129203|            101655|           100686|
|   mean|24.447977214151376|176.31540996507795|71.96177224241701|
| stddev| 5.693058519106263| 10.36921449417913|14.56364295943342|
|    min|                10|             127.0|             25.0|
|    max|                97|             226.0|            214.0|
+-------+------------------+------------------+-----------------+



### 5.2 Resumen

| Elemento | Resultado |
|---|---|
| RDD `deportista` | 67.786 registros de `deportista.csv` en seis particiones de entre 11.095 y 11.424 registros |
| RDD `deportista2` | 67.785 registros de `deportista2.csv` |
| RDD `deportistaTotal` | 135.571 registros (67.786 + 67.785), sin identificadores repetidos; nombres en mayúsculas |
| RDD `MayorEdad` | 123.244 deportistas de 18 años o más |
| RDD `Deportistas_mujer` | 33.981 deportistas con `genero = 2` |
| DataFrame `deportista` | 135.571 filas y siete columnas tipadas |

**Conclusiones**

- La unión conserva todos los registros de ambos archivos: 135.571 deportistas con identificadores únicos y continuos, que corresponden a los `deportista_id` usados en `resultados.csv`.
- Del total, 123.244 deportistas son mayores de edad, 5.959 tienen entre 10 y 17 años y 6.368 no tienen edad registrada. Estos últimos no se clasifican como menores.
- Las mujeres representan 33.981 registros (25,1 %) y los hombres 101.590 (74,9 %).
- La conversión a mayúsculas afectó solo a los nombres: ningún valor numérico cambió.
- En el DataFrame quedan sin dato 6.368 edades (4,7 %), 33.916 alturas (25,0 %) y 34.885 pesos (25,7 %). Entre los valores registrados, la edad promedio es de 24,4 años (rango de 10 a 97), la altura promedio de 176,3 cm y el peso promedio de 72,0 kg.
- La falta de altura o peso en cerca de un cuarto de los deportistas limitará el cálculo del IMC a los registros que tengan ambos valores.

## 6. DataFrames e integración — Sebastián

Este bloque continúa con `deportista`, construido por Yerko. No vuelve a leer ni reemplaza su entrada.
Se usa **Resultado como tabla central**, con una fila por registro de participación. Se conservan
los resultados sin correspondencia mediante joins izquierdos; no se deduplican participaciones.

Los nombres reales de los archivos difieren de los ejemplos de la consigna:

| DataFrame | Archivo | Clave | Lectura |
|---|---|---|---|
| Evento | evento.csv | cod_evento → evento_id | CSV con encabezado, coma |
| Resultado | resultados.csv | resultado_id | CSV con encabezado, punto y coma |
| Equipos | equipo.csv | id → equipo_id | CSV con encabezado, coma |
| Juego | juegos.json | juego_id | arreglo JSON, multiline |

Se declaran esquemas explícitos y lectura `FAILFAST`: una conversión inválida detiene el flujo.
`NA` en medalla se conserva como código de ausencia de medalla, distinto de un dato nulo.

**Anomalía de Evento:** 280 filas encapsulan el registro CSV completo en un único campo. Se retira esa capa usando `csv.reader`, se validan los tres campos y se entrega CSV normalizado a Spark. Se informa el número reparado y no se descartan filas ni se modifica la fuente.

`#N/A` en `deporte_id` se interpreta como nulo: es un atributo ausente, no la clave de Evento. Se muestran los eventos afectados y se conservan en la integración.

En Resultado hay nueve `evento_id = #N/A`: se leen como nulos y se conservan mediante el join izquierdo. No confundir `#N/A` (dato faltante) con `NA` en medalla (sin medalla).


In [16]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

# Algunas filas son un CSV completo encapsulado en un único campo CSV.
import csv
import io

def normalizar_evento(linea):
    campos = next(csv.reader([linea], strict=True))
    reparada = len(campos) == 1
    if reparada:
        campos = next(csv.reader([campos[0]], strict=True))
    if len(campos) != 3:
        raise ValueError(f"Evento con número de campos inválido: {linea!r}")
    salida = io.StringIO()
    csv.writer(salida, lineterminator="").writerow(campos)
    return salida.getvalue(), reparada

lineas_evento = sc.textFile(str(DIR_DATOS / "evento.csv"))
eventos_normalizados = lineas_evento.map(normalizar_evento)
print("Filas de Evento con doble encapsulado reparado:", eventos_normalizados.filter(lambda r: r[1]).count())
Evento = (spark.read.option("header", True).option("mode", "FAILFAST").option("escape", '\"')
          .schema("cod_evento INT, evento STRING, deporte_id INT")
          .option("nullValue", "#N/A")
          .csv(eventos_normalizados.map(lambda r: r[0]))
          .withColumnRenamed("cod_evento", "evento_id"))
Resultado = (spark.read.option("header", True).option("sep", ";").option("mode", "FAILFAST")
             .schema("resultado_id INT, medalla STRING, deportista_id INT, juego_id INT, evento_id INT")
             .option("nullValue", "#N/A")
             .csv(str(DIR_DATOS / "resultados.csv")))
Equipos = (spark.read.option("header", True).option("mode", "FAILFAST")
           .schema("id INT, equipo STRING, sigla STRING")
           .csv(str(DIR_DATOS / "equipo.csv"))
           .withColumnRenamed("id", "equipo_id"))
Juego_fuente = (spark.read.option("multiLine", True).option("mode", "FAILFAST")
                .schema("juego_id INT, ano STRING, temporada INT, ciudad STRING")
                .json(str(DIR_DATOS / "juegos.json")))
Juego_fuente.show(5, truncate=False)
print("Eventos sin deporte_id:", Evento.filter(F.col("deporte_id").isNull()).count())
Evento.filter(F.col("deporte_id").isNull()).show(truncate=False)


Filas de Evento con doble encapsulado reparado: 280


+--------+-----------+---------+------+
|juego_id|ano        |temporada|ciudad|
+--------+-----------+---------+------+
|1       |1896 Verano|1896     |Verano|
|2       |1900 Verano|1900     |Verano|
|3       |1904 Verano|1904     |Verano|
|4       |1906 Verano|1906     |Verano|
|5       |1908 Verano|1908     |Verano|
+--------+-----------+---------+------+
only showing top 5 rows
Eventos sin deporte_id: 2


+---------+------------------------------------------+----------+
|evento_id|evento                                    |deporte_id|
+---------+------------------------------------------+----------+
|703      |Sailing Mixed 20+ Ton                     |NULL      |
|743      |Motorboating Mixed B-Class (Under 60 Feet)|NULL      |
+---------+------------------------------------------+----------+



### 6.1 Corrección semántica de Juegos

La fuente trae `ano = "1896 Verano"`, `temporada = 1896` y `ciudad = "Verano"`.
Se comprueba la coherencia de esos valores en **todas** las filas antes de renombrarlos:
`ano` pasa a `edicion`, `temporada` a `anio`, y `ciudad` a `temporada`.
Se conservan además los tres campos originales con sufijo `_fuente` para trazabilidad.
No hay una ciudad sede utilizable en este archivo; no se inventa una.

In [17]:
condicion_juego = (
    F.col("ciudad").isin("Verano", "Invierno")
    & F.col("temporada").isNotNull()
    & (F.col("ano") == F.concat_ws(" ", F.col("temporada"), F.col("ciudad")))
)
assert Juego_fuente.filter(~F.coalesce(condicion_juego, F.lit(False))).count() == 0, "Revisar estructura de Juegos"
Juego = Juego_fuente.select(
    "juego_id", F.col("ano").alias("edicion"), F.col("temporada").alias("anio"),
    F.col("ciudad").alias("temporada"), F.col("ano").alias("ano_fuente"),
    F.col("temporada").alias("temporada_fuente"), F.col("ciudad").alias("ciudad_fuente")
)
conteos_fuentes = {}
for nombre_tabla, tabla in [("Evento", Evento), ("Resultado", Resultado), ("Equipos", Equipos), ("Juego", Juego)]:
    conteos_fuentes[nombre_tabla] = tabla.count()
    print(nombre_tabla, "— filas:", conteos_fuentes[nombre_tabla])
    tabla.printSchema()
    tabla.show(5, truncate=False)
Resultado.groupBy("medalla").count().orderBy("medalla").show()

Evento — filas: 765
root
 |-- evento_id: integer (nullable = true)
 |-- evento: string (nullable = true)
 |-- deporte_id: integer (nullable = true)

+---------+--------------------------------+----------+
|evento_id|evento                          |deporte_id|
+---------+--------------------------------+----------+
|1        |Basketball Men's Basketball     |1         |
|2        |Judo Men's Extra-Lightweight    |2         |
|3        |Football Men's Football         |3         |
|4        |Tug-Of-War Men's Tug-Of-War     |4         |
|5        |Speed Skating Women's 500 metres|5         |
+---------+--------------------------------+----------+
only showing top 5 rows


Resultado — filas: 271116
root
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- juego_id: integer (nullable = true)
 |-- evento_id: integer (nullable = true)

+------------+-------+-------------+--------+---------+
|resultado_id|medalla|deportista_id|juego_id|evento_id|
+------------+-------+-------------+--------+---------+
|1           |NA     |1            |39      |1        |
|2           |NA     |2            |49      |2        |
|3           |NA     |3            |7       |3        |
|4           |Gold   |4            |2       |4        |
|5           |NA     |5            |36      |5        |
+------------+-------+-------------+--------+---------+
only showing top 5 rows
Equipos — filas: 1184
root
 |-- equipo_id: integer (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: string (nullable = true)

+---------+---------------------+-----+
|equipo_id|equipo               |sigla|
+-

Juego — filas: 51
root
 |-- juego_id: integer (nullable = true)
 |-- edicion: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- temporada: string (nullable = true)
 |-- ano_fuente: string (nullable = true)
 |-- temporada_fuente: integer (nullable = true)
 |-- ciudad_fuente: string (nullable = true)

+--------+-----------+----+---------+-----------+----------------+-------------+
|juego_id|edicion    |anio|temporada|ano_fuente |temporada_fuente|ciudad_fuente|
+--------+-----------+----+---------+-----------+----------------+-------------+
|1       |1896 Verano|1896|Verano   |1896 Verano|1896            |Verano       |
|2       |1900 Verano|1900|Verano   |1900 Verano|1900            |Verano       |
|3       |1904 Verano|1904|Verano   |1904 Verano|1904            |Verano       |
|4       |1906 Verano|1906|Verano   |1906 Verano|1906            |Verano       |
|5       |1908 Verano|1908|Verano   |1908 Verano|1908            |Verano       |
+--------+-----------+----+-------

+-------+------+
|medalla| count|
+-------+------+
| Bronze| 13295|
|   Gold| 13372|
|     NA|231333|
| Silver| 13116|
+-------+------+



### 6.2 Controles de claves y correspondencias

Una clave repetida en una dimensión multiplicaría resultados. Se detiene la integración si
hay claves nulas o duplicadas, incluido `resultado_id`. Los joins anti permiten contar las
referencias sin correspondencia antes de unir. Un equipo ausente se conserva como nulo:
no se asigna a un país inventado ni se elimina la participación.

In [18]:
def verificar_clave(tabla, clave, nombre_tabla):
    resumen = tabla.agg(F.count("*").alias("filas"), F.count(clave).alias("presentes"),
                       F.countDistinct(clave).alias("distintas")).first()
    print(nombre_tabla, clave, resumen.asDict())
    assert resumen.filas == resumen.presentes == resumen.distintas, f"Clave no válida en {nombre_tabla}"

for tabla, clave, nombre_tabla in [
    (deportista, "deportista_id", "deportista"), (Evento, "evento_id", "Evento"),
    (Equipos, "equipo_id", "Equipos"), (Juego, "juego_id", "Juego"),
    (Resultado, "resultado_id", "Resultado")
]:
    verificar_clave(tabla, clave, nombre_tabla)

for nombre_control, izquierda, derecha, clave in [
    ("Resultados sin deportista", Resultado, deportista, "deportista_id"),
    ("Resultados sin evento", Resultado, Evento, "evento_id"),
    ("Resultados sin juego", Resultado, Juego, "juego_id"),
    ("Deportistas sin equipo", deportista, Equipos, "equipo_id")
]:
    sin_correspondencia = izquierda.join(derecha.select(clave), clave, "left_anti")
    print(nombre_control, ":", sin_correspondencia.count())
    sin_correspondencia.select(clave).distinct().orderBy(clave).show(5)

deportista deportista_id {'filas': 135571, 'presentes': 135571, 'distintas': 135571}


Evento evento_id {'filas': 765, 'presentes': 765, 'distintas': 765}
Equipos equipo_id {'filas': 1184, 'presentes': 1184, 'distintas': 1184}


Juego juego_id {'filas': 51, 'presentes': 51, 'distintas': 51}


Resultado resultado_id {'filas': 271116, 'presentes': 271116, 'distintas': 271116}


Resultados sin deportista : 0


+-------------+
|deportista_id|
+-------------+
+-------------+



Resultados sin evento : 9


+---------+
|evento_id|
+---------+
|     NULL|
+---------+

Resultados sin juego : 0


+--------+
|juego_id|
+--------+
+--------+



Deportistas sin equipo : 143


+---------+
|equipo_id|
+---------+
|        0|
+---------+



### 6.3 Integración y optimización

Se seleccionan únicamente las columnas necesarias de cada entrada (más los campos de
trazabilidad de Juegos). Los esquemas explícitos evitan inferencia y mantienen claves compatibles.
Las dimensiones Evento, Equipos y Juego son archivos pequeños (menos de 100 KB cada uno);
se verifica también que cada dimensión tenga menos de 5.000 filas antes de usar `broadcast`.
El plan físico permite observar los joins. No se afirma una mejora de tiempo sin una comparación medida.

El resultado se persiste con `MEMORY_AND_DISK` porque se reutiliza en la inspección y en las
consultas posteriores de José. Se materializa con `count`. Se debe liberar con
`df_integrado.unpersist()` **después** de terminar esas consultas.

In [19]:
for nombre_tabla, archivo in [("Evento", "evento.csv"), ("Equipos", "equipo.csv"), ("Juego", "juegos.json")]:
    tamano = (DIR_DATOS / archivo).stat().st_size
    print(nombre_tabla, "— bytes de fuente:", tamano, "— filas:", conteos_fuentes[nombre_tabla])
    assert tamano < 100_000 and conteos_fuentes[nombre_tabla] < 5_000, "Reevaluar uso de broadcast"

df_union = (Resultado.select("resultado_id", "medalla", "deportista_id", "juego_id", "evento_id")
    .join(deportista.select("deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"), "deportista_id", "left")
    .join(F.broadcast(Evento.select("evento_id", "evento", "deporte_id")), "evento_id", "left")
    .join(F.broadcast(Equipos.select("equipo_id", "equipo", "sigla")), "equipo_id", "left")
    .join(F.broadcast(Juego), "juego_id", "left"))
df_union.explain(mode="simple")

Evento — bytes de fuente: 35144 — filas: 765
Equipos — bytes de fuente: 23606 — filas: 1184
Juego — bytes de fuente: 5232 — filas: 51
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [juego_id#322, equipo_id#15, evento_id#323, deportista_id#321, resultado_id#319, medalla#320, nombre#10, genero#11, edad#12, altura#13, peso#14, evento#316, deporte_id#317, equipo#325, sigla#326, edicion#379, anio#380, temporada#381, ano_fuente#382, temporada_fuente#383, ciudad_fuente#384]
   +- BroadcastHashJoin [juego_id#322], [juego_id#329], LeftOuter, BuildRight, false, false
      :- Project [equipo_id#15, evento_id#323, deportista_id#321, resultado_id#319, medalla#320, juego_id#322, nombre#10, genero#11, edad#12, altura#13, peso#14, evento#316, deporte_id#317, equipo#325, sigla#326]
      :  +- BroadcastHashJoin [equipo_id#15], [equipo_id#328], LeftOuter, BuildRight, false, false
      :     :- Project [evento_id#323, deportista_id#321, resultado_id#319, medalla#320, juego_id#322, n

## 7. Paralelismo — Sebastián

`repartition(5)` redistribuye los registros mediante shuffle. Las cinco particiones son
particiones lógicas, no cinco máquinas; no implican por sí solas mayor velocidad local.
Se verifica la distribución después de materializar el DataFrame persistido.

In [20]:
df_integrado = df_union.repartition(5).persist(StorageLevel.MEMORY_AND_DISK)
n_integrado = df_integrado.count()
n_particiones = df_integrado.rdd.getNumPartitions()
filas_por_particion = df_integrado.rdd.mapPartitions(lambda filas: [sum(1 for _ in filas)]).collect()
print("Filas de Resultado:", conteos_fuentes["Resultado"])
print("Filas integradas:", n_integrado)
print("Particiones:", n_particiones)
print("Filas por partición:", filas_por_particion)
print("Persistencia:", df_integrado.storageLevel)
assert n_particiones == 5
assert sum(filas_por_particion) == n_integrado == conteos_fuentes["Resultado"]
assert df_integrado.select("resultado_id").distinct().count() == n_integrado, "Se multiplicaron resultados"

Filas de Resultado: 271116
Filas integradas: 271116
Particiones: 5
Filas por partición: [54222, 54224, 54223, 54222, 54225]
Persistencia: Disk Memory Serialized 1x Replicated


## 8. Inspección y entrega a José — Sebastián

`df_integrado` queda disponible para IMC, `Descripción_sexo` y agregaciones SQL.
`genero` mantiene 1 = hombre, 2 = mujer; `edad` está en años, `altura` en centímetros y
`peso` en kg según el contrato del bloque anterior. Los ceros de esas tres medidas ya fueron
convertidos a nulo por Yerko. `temporada` contiene Verano/Invierno y `anio` contiene el año.
`medalla` conserva Gold, Silver, Bronze y NA (sin medalla).

La unidad del análisis es la participación: una persona puede aparecer varias veces.
Los conteos de medallas por registro no equivalen necesariamente al medallero oficial de
pruebas por equipos. Se conserva `equipo_id` incluso cuando no existe en Equipos;
José deberá mostrar explícitamente esos casos si agrupa por nombre de equipo.

In [21]:
print("Número de filas:", n_integrado)
print("Tipo del objeto:", type(df_integrado))
print("Tipos de las columnas:", df_integrado.dtypes)
df_integrado.printSchema()
df_integrado.orderBy("resultado_id").show(10, truncate=False)
df_integrado.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["nombre", "evento", "equipo", "temporada", "edad", "altura", "peso"]
]).show()
print("Participaciones sin equipo identificado:", df_integrado.filter(F.col("equipo").isNull()).count())
df_integrado.filter(F.col("equipo").isNull()).select(
    "resultado_id", "deportista_id", "equipo_id", "equipo"
).orderBy("resultado_id").show(5)

Número de filas: 271116
Tipo del objeto: <class 'pyspark.sql.classic.dataframe.DataFrame'>
Tipos de las columnas: [('juego_id', 'int'), ('equipo_id', 'int'), ('evento_id', 'int'), ('deportista_id', 'int'), ('resultado_id', 'int'), ('medalla', 'string'), ('nombre', 'string'), ('genero', 'int'), ('edad', 'int'), ('altura', 'double'), ('peso', 'double'), ('evento', 'string'), ('deporte_id', 'int'), ('equipo', 'string'), ('sigla', 'string'), ('edicion', 'string'), ('anio', 'int'), ('temporada', 'string'), ('ano_fuente', 'string'), ('temporada_fuente', 'int'), ('ciudad_fuente', 'string')]
root
 |-- juego_id: integer (nullable = true)
 |-- equipo_id: integer (nullable = true)
 |-- evento_id: integer (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = tru

+------+------+------+---------+----+------+-----+
|nombre|evento|equipo|temporada|edad|altura| peso|
+------+------+------+---------+----+------+-----+
|     0|     9|   200|        0|9474| 60171|62875|
+------+------+------+---------+----+------+-----+

Participaciones sin equipo identificado: 200


+------------+-------------+---------+------+
|resultado_id|deportista_id|equipo_id|equipo|
+------------+-------------+---------+------+
|         654|          378|        0|  NULL|
|        5946|         3341|        0|  NULL|
|        6147|         3459|        0|  NULL|
|        6148|         3459|        0|  NULL|
|        6149|         3459|        0|  NULL|
+------------+-------------+---------+------+
only showing top 5 rows


### Cierre del bloque de Sebastián

Los controles anteriores verifican que cada resultado sigue apareciendo exactamente una vez,
que no se pierden referencias sin equipo y que la tabla tiene cinco particiones.
La anomalía semántica de Juegos queda corregida y trazable sin modificar el archivo original.
Las celdas siguientes corresponden al bloque de José: columnas calculadas y cuatro agregaciones SQL.